# HESTON-X: A Parallel Exotic-Options Pricing & Greeks Engine

**Problem.** A derivatives desk holds a book of path-dependent exotic options (European, Asian, barrier, lookback) on one underlying whose volatility is *stochastic* (Heston model). No closed form exists for the exotics, so the book must be priced and risk-managed (Delta, Vega) by Monte Carlo. Doing this for 5 market scenarios x millions of paths x 252 monitoring dates is the bottleneck, so we parallelise it.

**What this notebook does**
1. Heston full-truncation Euler simulation with antithetic variates.
2. Validation of the engine against the **semi-analytic Heston European price** (Fourier inversion).
3. Four parallel implementations: serial NumPy, `multiprocessing` (data parallelism over path chunks), two-level task parallelism (scenario x chunk), and Numba `prange` (shared-memory threads, fused kernel).
4. Statistically safe parallel RNG (`SeedSequence.spawn`) so results do not depend on worker count.
5. Greeks via bump-and-reprice with **common random numbers**.
6. Performance study: speedup, efficiency, Karp-Flatt serial fraction, Amdahl bound.

Run all cells top to bottom (Runtime > Run all). Default sizes finish in a few minutes on free Colab.

In [ ]:
import os, time, math
import numpy as np
import matplotlib.pyplot as plt
from multiprocessing import Pool
from scipy.integrate import quad
import numba as nb
from numba import njit, prange

NCPU = os.cpu_count()
print("CPU cores available:", NCPU)

# ---------------- Market & model parameters ----------------
S0, r, q = 100.0, 0.05, 0.01
V0, KAPPA, THETA, XI, RHO = 0.04, 2.0, 0.04, 0.5, -0.7   # Heston
T, N_STEPS = 1.0, 252

# ---------------- The option book (strike, barrier) ----------------
K_EUR, K_ASIAN, K_BAR, B_UP, K_LOOK = 100.0, 100.0, 100.0, 130.0, 100.0
INSTR = ["European call", "Asian arith. call", "Up-and-out call", "Fixed lookback call"]
NI = len(INSTR)
PARAMS = np.array([S0, r, q, V0, KAPPA, THETA, XI, RHO, T,
                   K_EUR, K_ASIAN, K_BAR, B_UP, K_LOOK], dtype=np.float64)

## 1. Semi-analytic Heston European call (ground truth)
Gil-Pelaez inversion with the numerically stable "little trap" characteristic function. Used only to validate the Monte Carlo engine.

In [ ]:
def heston_call(S, K, T, r, q, v0, kappa, theta, xi, rho):
    x, lnK = np.log(S), np.log(K)
    def P(j):
        u = 0.5 if j == 1 else -0.5
        b = kappa - rho * xi if j == 1 else kappa
        def integrand(phi):
            i = 1j
            t = b - rho * xi * i * phi
            d = np.sqrt(t * t - xi**2 * (2 * u * i * phi - phi**2))
            g = (t - d) / (t + d)
            e = np.exp(-d * T)
            C = (r - q) * i * phi * T + kappa * theta / xi**2 * (
                (t - d) * T - 2 * np.log((1 - g * e) / (1 - g)))
            D = (t - d) / xi**2 * (1 - e) / (1 - g * e)
            return (np.exp(C + D * v0 + i * phi * x - i * phi * lnK) / (i * phi)).real
        return 0.5 + quad(integrand, 1e-8, 200, limit=500)[0] / np.pi
    return S * np.exp(-q * T) * P(1) - K * np.exp(-r * T) * P(2)

REF_EUR = heston_call(S0, K_EUR, T, r, q, V0, KAPPA, THETA, XI, RHO)
print(f"Semi-analytic Heston European call: {REF_EUR:.5f}")

## 2. Monte Carlo kernels

**Kernel A (NumPy, vectorised over paths).** Simulates `n_paths` antithetic pairs in memory-bounded batches and returns, for each instrument, the **sum and sum of squares** of discounted payoffs. Returning sufficient statistics (not paths) keeps inter-process communication tiny, which is key for good speedup.

**Kernel B (Numba).** One fused loop per path, no temporary arrays, parallelised with `prange`.

In [ ]:
def mc_chunk(args):
    """Price the whole book on n_paths antithetic pairs. Returns (NI,2) array [sum, sum_sq]."""
    p, n_paths, n_steps, seed, batch = args
    S_0, r_, q_, v_0, kap, th, xi_, rho_, T_, kE, kA, kB, Bu, kL = p
    rng = np.random.default_rng(seed)
    dt = T_ / n_steps; sdt = math.sqrt(dt); disc = math.exp(-r_ * T_)
    out = np.zeros((NI, 2))
    done = 0
    while done < n_paths:
        n = min(batch, n_paths - done); done += n
        S = np.full(2 * n, S_0); v = np.full(2 * n, v_0)       # [first n: +Z, last n: -Z]
        sumS = np.zeros(2 * n); smax = S.copy(); alive = np.ones(2 * n, bool)
        for _ in range(n_steps):
            z1 = rng.standard_normal(n); zp = rng.standard_normal(n)
            z2 = rho_ * z1 + math.sqrt(1 - rho_**2) * zp
            z1 = np.concatenate((z1, -z1)); z2 = np.concatenate((z2, -z2))   # antithetic
            vp = np.maximum(v, 0.0)
            S = S * np.exp((r_ - q_ - 0.5 * vp) * dt + np.sqrt(vp) * sdt * z1)
            v = v + kap * (th - vp) * dt + xi_ * np.sqrt(vp) * sdt * z2     # full truncation
            sumS += S; smax = np.maximum(smax, S); alive &= (S < Bu)
        pay = np.empty((NI, 2 * n))
        pay[0] = np.maximum(S - kE, 0)
        pay[1] = np.maximum(sumS / n_steps - kA, 0)
        pay[2] = np.maximum(S - kB, 0) * alive
        pay[3] = np.maximum(smax - kL, 0)
        pay *= disc
        # antithetic pairing: average pair first, then accumulate (valid i.i.d. samples)
        pair = 0.5 * (pay[:, :n] + pay[:, n:])
        out[:, 0] += pair.sum(1); out[:, 1] += (pair**2).sum(1)
    return out

@njit(parallel=True, fastmath=True, cache=True)
def mc_numba(p, n_blocks, paths_per_block, n_steps, seed):
    S_0, r_, q_, v_0, kap, th, xi_, rho_, T_, kE, kA, kB, Bu, kL = p
    dt = T_ / n_steps; sdt = math.sqrt(dt); disc = math.exp(-r_ * T_)
    c = math.sqrt(1 - rho_ * rho_)
    res = np.zeros((n_blocks, 4, 2))
    for b in prange(n_blocks):
        np.random.seed(seed + b)              # per-block stream -> deterministic for any thread count
        for _ in range(paths_per_block):
            Sa = S_0; Sb = S_0; va = v_0; vb = v_0
            suma = 0.0; sumb = 0.0; ma = S_0; mb = S_0; al_a = 1.0; al_b = 1.0
            for _t in range(n_steps):
                z1 = np.random.standard_normal(); zp = np.random.standard_normal()
                z2 = rho_ * z1 + c * zp
                vpa = max(va, 0.0); vpb = max(vb, 0.0)
                Sa *= math.exp((r_ - q_ - 0.5 * vpa) * dt + math.sqrt(vpa) * sdt * z1)
                Sb *= math.exp((r_ - q_ - 0.5 * vpb) * dt - math.sqrt(vpb) * sdt * z1)
                va += kap * (th - vpa) * dt + xi_ * math.sqrt(vpa) * sdt * z2
                vb += kap * (th - vpb) * dt - xi_ * math.sqrt(vpb) * sdt * z2
                suma += Sa; sumb += Sb
                ma = max(ma, Sa); mb = max(mb, Sb)
                if Sa >= Bu: al_a = 0.0
                if Sb >= Bu: al_b = 0.0
            x0 = 0.5 * (max(Sa - kE, 0) + max(Sb - kE, 0)) * disc
            x1 = 0.5 * (max(suma / n_steps - kA, 0) + max(sumb / n_steps - kA, 0)) * disc
            x2 = 0.5 * (max(Sa - kB, 0) * al_a + max(Sb - kB, 0) * al_b) * disc
            x3 = 0.5 * (max(ma - kL, 0) + max(mb - kL, 0)) * disc
            res[b, 0, 0] += x0; res[b, 0, 1] += x0 * x0
            res[b, 1, 0] += x1; res[b, 1, 1] += x1 * x1
            res[b, 2, 0] += x2; res[b, 2, 1] += x2 * x2
            res[b, 3, 0] += x3; res[b, 3, 1] += x3 * x3
    return res.sum(axis=0)

def summarize(stats, n_pairs):
    mean = stats[:, 0] / n_pairs
    var = np.maximum(stats[:, 1] / n_pairs - mean**2, 0)
    return mean, np.sqrt(var / n_pairs)          # price, standard error

## 3. Parallel drivers (reproducible RNG)
`SeedSequence.spawn` gives each *chunk* its own statistically independent stream. We split the work into a **fixed number of chunks** independent of worker count, so the answer is bit-for-bit identical for 1 or N workers.

In [ ]:
N_PAIRS = 200_000          # antithetic pairs (= 400k paths).  Increase for tighter error bars.
N_CHUNKS = 32
BATCH = 5_000

def make_tasks(params, n_pairs, n_chunks, root_seed=2024, batch=BATCH):
    seeds = np.random.SeedSequence(root_seed).spawn(n_chunks)
    per = n_pairs // n_chunks
    return [(params, per, N_STEPS, s, batch) for s in seeds], per * n_chunks

def price_serial(params, n_pairs=N_PAIRS):
    tasks, tot = make_tasks(params, n_pairs, N_CHUNKS)
    st = sum(mc_chunk(t) for t in tasks)
    return summarize(st, tot)

def price_pool(params, n_pairs=N_PAIRS, workers=NCPU):
    tasks, tot = make_tasks(params, n_pairs, N_CHUNKS)
    with Pool(workers) as pool:
        st = sum(pool.imap_unordered(mc_chunk, tasks))
    return summarize(st, tot)

def price_numba(params, n_pairs=N_PAIRS, threads=NCPU, n_blocks=N_CHUNKS):
    nb.set_num_threads(threads)
    st = mc_numba(params, n_blocks, n_pairs // n_blocks, N_STEPS, 12345)
    return summarize(st, (n_pairs // n_blocks) * n_blocks)

mc_numba(PARAMS, 2, 10, 10, 1)   # JIT warm-up (excluded from timings)

## 4. Correctness check against the analytic price

In [ ]:
t0 = time.perf_counter(); pr, se = price_pool(PARAMS); t_pool = time.perf_counter() - t0
t0 = time.perf_counter(); pn, sn = price_numba(PARAMS); t_nb = time.perf_counter() - t0

print(f"{'Instrument':22s}{'Pool MC':>10s}{'+/-SE':>9s}{'Numba MC':>11s}{'+/-SE':>9s}")
for i, n in enumerate(INSTR):
    print(f"{n:22s}{pr[i]:10.4f}{se[i]:9.4f}{pn[i]:11.4f}{sn[i]:9.4f}")
z = (pr[0] - REF_EUR) / se[0]
print(f"\nAnalytic European: {REF_EUR:.4f} | MC error = {pr[0]-REF_EUR:+.4f} ({z:+.2f} std errors; |z|<3 is a pass)")
print(f"Pool time {t_pool:.1f}s | Numba time {t_nb:.1f}s")
# sanity: knock-out <= vanilla, lookback >= vanilla
assert pr[2] <= pr[0] and pr[3] >= pr[0]

## 5. Two-level parallel Greeks (task parallelism x data parallelism)
Five scenarios (base, S0 +/- 1%, V0 +/- 10%) are repriced with the **same seeds** (common random numbers) so noise cancels in the finite differences. All `scenario x chunk` tasks go into one pool queue, which balances load far better than parallelising only one level.

* Delta = (P(S+) - P(S-)) / (2 dS)
* Vega-V0 (sensitivity to initial variance) = (P(V+) - P(V-)) / (2 dV)

In [ ]:
def scenario_params(base):
    sc = {"base": base.copy()}
    for name, idx, f in [("S+", 0, 1.01), ("S-", 0, 0.99), ("V+", 3, 1.10), ("V-", 3, 0.90)]:
        p = base.copy(); p[idx] *= f; sc[name] = p
    return sc

def greeks_parallel(base, n_pairs=N_PAIRS, workers=NCPU):
    sc = scenario_params(base)
    tasks, tot = [], None
    for name, p in sc.items():
        t, tot = make_tasks(p, n_pairs, N_CHUNKS)          # identical seeds across scenarios = CRN
        tasks += [(name, x) for x in t]
    with Pool(workers) as pool:
        outs = pool.map(_run_named, tasks, chunksize=2)
    agg = {n: np.zeros((NI, 2)) for n in sc}
    for n, o in outs: agg[n] += o
    price = {n: summarize(agg[n], tot)[0] for n in sc}
    dS = 0.02 * base[0]; dV = 0.2 * base[3]
    delta = (price["S+"] - price["S-"]) / dS
    vega = (price["V+"] - price["V-"]) / dV
    return price["base"], delta, vega

def _run_named(t):
    return t[0], mc_chunk(t[1])

t0 = time.perf_counter()
P, D, Vg = greeks_parallel(PARAMS)
print(f"Greeks computed in {time.perf_counter()-t0:.1f}s on {NCPU} worker(s)\n")
print(f"{'Instrument':22s}{'Price':>9s}{'Delta':>9s}{'dP/dV0':>10s}")
for i, n in enumerate(INSTR):
    print(f"{n:22s}{P[i]:9.4f}{D[i]:9.4f}{Vg[i]:10.3f}")

## 6. Performance study: speedup, efficiency, Karp-Flatt, Amdahl
Strong scaling on a fixed problem. Free Colab usually exposes only 2 cores, so we also run oversubscribed worker counts to show where scaling saturates. Each point is the best of 2 runs.

In [ ]:
def best_time(fn, reps=2):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return min(ts)

BENCH_PAIRS = 100_000
workers_list = sorted(set([1, 2, 4, NCPU, 2 * NCPU]))

print("Timing serial baseline ..."); T1 = best_time(lambda: price_serial(PARAMS, BENCH_PAIRS), 1)
rows = []
for w in workers_list:
    tp = best_time(lambda: price_pool(PARAMS, BENCH_PAIRS, w))
    tn = best_time(lambda: price_numba(PARAMS, BENCH_PAIRS, min(w, NCPU)))   # threads capped at cores
    rows.append((w, tp, tn))
    print(f"workers={w:2d}  pool={tp:6.2f}s  numba={tn:6.2f}s")

print(f"\nSerial NumPy baseline: {T1:.2f}s")
print(f"{'p':>3s}{'Pool S':>9s}{'Pool E':>9s}{'KF e':>8s}{'Numba S':>10s}")
for w, tp, tn in rows:
    S = T1 / tp; E = S / w
    kf = ((1 / S - 1 / w) / (1 - 1 / w)) if w > 1 else float('nan')   # Karp-Flatt serial fraction
    print(f"{w:3d}{S:9.2f}{E:9.2f}{kf:8.3f}{T1/tn:10.2f}")

In [ ]:
ws = np.array([r[0] for r in rows]); Sp = T1 / np.array([r[1] for r in rows]); Sn = T1 / np.array([r[2] for r in rows])
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(ws, ws, 'k--', label='Ideal')
ax[0].plot(ws, Sp, 'o-', label='multiprocessing (NumPy)')
ax[0].plot(ws, Sn, 's-', label='Numba prange (vs NumPy serial)')
ax[0].axvline(NCPU, color='gray', ls=':'); ax[0].text(NCPU, 0.5, ' physical cores', color='gray')
ax[0].set_xlabel('workers / threads'); ax[0].set_ylabel('speedup vs serial NumPy'); ax[0].set_title('Strong scaling'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ws, Sp / ws, 'o-'); ax[1].set_ylim(0, 1.1); ax[1].set_xlabel('workers'); ax[1].set_ylabel('parallel efficiency'); ax[1].set_title('Efficiency (multiprocessing)'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 7. Convergence: the payoff of going parallel
Standard error falls as 1/sqrt(N), so halving the error needs 4x the paths. Parallel hardware turns that cost into wall-clock time you can afford.

In [ ]:
sizes = [10_000, 40_000, 160_000, 640_000]
errs = []
for n in sizes:
    p_, s_ = price_numba(PARAMS, n)
    errs.append(s_)
errs = np.array(errs)
plt.figure(figsize=(6, 4.5))
for i, nme in enumerate(INSTR): plt.loglog(sizes, errs[:, i], 'o-', label=nme)
plt.loglog(sizes, errs[0, 0] * np.sqrt(sizes[0] / np.array(sizes)), 'k--', label='1/sqrt(N)')
plt.xlabel('antithetic pairs'); plt.ylabel('standard error'); plt.legend(); plt.grid(alpha=.3, which='both'); plt.show()

## Report guide
**Analysis questions to answer in your write-up**
1. Why is returning `(sum, sum_sq)` per chunk better than returning simulated paths? (Communication-to-computation ratio.)
2. Why does Numba `prange` beat `multiprocessing` even at the same core count? (No IPC/pickling, no temporaries, fused loop, SIMD/`fastmath`.)
3. Use the Karp-Flatt serial fraction to explain why efficiency drops, and compute the Amdahl ceiling `1/e`.
4. What happens when workers > cores (oversubscription, context switches)?
5. Why must RNG streams be spawned per chunk rather than seeding every worker with the same seed? Show that the result is identical for 1 and N workers.
6. Why use common random numbers in bump-and-reprice? Rerun Greeks with different seeds per scenario and compare noise.
7. The up-and-out barrier payoff is discontinuous, so bump Delta is noisy. Propose a fix (smoothing, Brownian-bridge barrier correction, likelihood-ratio method).

**Extensions (for extra marks)**
* Add Longstaff-Schwartz American options (parallel regression is the hard part).
* Add portfolio VaR/CVaR with nested simulation.
* Port the kernel to GPU with `numba.cuda` or CuPy (Runtime > Change runtime type > T4 GPU).
* Weak-scaling study (paths grow with workers).